# Customer Churn Prediction — Supervised Learning

## Introduction

The goal of this project is to predict customer churn for **Beta Bank** using historical customer behavior data. The bank has noticed that customers are leaving each month, and retaining existing customers is more cost-effective than acquiring new ones.

The primary evaluation metric is **F1 score**, with a minimum threshold of **0.59**. **AUC-ROC** is also tracked to assess overall ranking quality. The project covers data preprocessing, class imbalance handling, model training and comparison, and final held-out test evaluation.

## Dataset

**File:** `datasets/Churn.csv` — 10,000 customer records

| Feature | Description |
|---|---|
| `CreditScore` | Customer credit score |
| `Geography` | Country of residence |
| `Gender` | Customer gender |
| `Age` | Customer age |
| `Tenure` | Years as a bank customer |
| `Balance` | Account balance |
| `NumOfProducts` | Number of bank products held |
| `HasCrCard` | Whether the customer has a credit card |
| `IsActiveMember` | Whether the customer is an active member |
| `EstimatedSalary` | Estimated annual salary |
| `Exited` | Target: 1 = churned, 0 = retained (~20% churn rate) |

## 1. Data Loading & Exploration

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils import resample

data = pd.read_csv('/datasets/Churn.csv')
print(data.info())
print(data.describe())
print(data['Exited'].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB
None
0    0.7963
1    0.2037
Name: Exited, dtype: float64


## 2. Preprocessing & Train / Validation / Test Split

The following preprocessing steps were applied:

- **Dropped irrelevant columns:** `RowNumber`, `CustomerId`, `Surname` — these contain no predictive signal and may introduce noise.
- **Imputed missing values** in `Tenure` using the median to preserve the feature's central tendency.
- **One-hot encoded** categorical variables (`Geography`, `Gender`) to make them compatible with ML models.
- **Scaled numerical features** using `StandardScaler` — fit on the training set only, then applied to validation and test sets to prevent data leakage.

The dataset is split **60 / 20 / 20** (train / validation / test) using stratified sampling to preserve class proportions.

In [2]:
# Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Fill missing values
data['Tenure'] = data['Tenure'].fillna(data['Tenure'].median())

# Encode categorical variables
data = pd.get_dummies(data, drop_first=True)

# Separate features and target
features = data.drop('Exited', axis=1)
target = data['Exited']

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    features, target, test_size=0.20, random_state=12345, stratify=target
)

# Second split: train/validation from remaining 80% (60/20/20 overall)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=12345, stratify=y_temp
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

## 3. Handling Class Imbalance

The dataset has a significant class imbalance (~80% retained, ~20% churned). Two approaches are tested and compared:

1. **Upsampling** — minority class is resampled with replacement to match majority class size
2. **Class weighting** — `class_weight='balanced'` applied directly in the Random Forest estimator

A baseline model (no imbalance handling) is also trained for comparison.

In [3]:
# Handle Class Imbalance (Upsampling)

train_df = pd.DataFrame(X_train_scaled)
train_df['Exited'] = y_train.values

majority = train_df[train_df['Exited'] == 0]
minority = train_df[train_df['Exited'] == 1]

minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=12345)
upsampled = pd.concat([majority, minority_upsampled])

X_train_up = upsampled.drop('Exited', axis=1)
y_train_up = upsampled['Exited']

## 4. Model Training & Hyperparameter Tuning

Two models are trained: **Logistic Regression** and **Random Forest**. Each is evaluated under different imbalance-handling strategies, with hyperparameters tuned via `GridSearchCV`.

### 4a. Logistic Regression

In [4]:
# Model 1: Logistic Regression

print("\n--- Logistic Regression ---")

# Base model
log_model = LogisticRegression(random_state=12345, max_iter=1000)
log_model.fit(X_train_scaled, y_train)
log_preds = log_model.predict(X_valid_scaled)

print("Base Logistic Regression F1:", f1_score(y_valid, log_preds))
print("Base Logistic Regression AUC-ROC:", roc_auc_score(y_valid, log_model.predict_proba(X_valid_scaled)[:, 1]))

# Upsampled model
log_model_up = LogisticRegression(random_state=12345, max_iter=1000)
log_model_up.fit(X_train_up, y_train_up)
log_preds_up = log_model_up.predict(X_valid_scaled)

print("Upsampled Logistic Regression F1:", f1_score(y_valid, log_preds_up))
print("Upsampled Logistic Regression AUC-ROC:", roc_auc_score(y_valid, log_model_up.predict_proba(X_valid_scaled)[:, 1]))

# Hyperparameter tuning
param_grid_lr = {'C': [0.1, 1, 10]}
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=12345),
                       param_grid_lr, scoring='f1', cv=3, n_jobs=-1)
grid_lr.fit(X_train_up, y_train_up)
best_log_model = grid_lr.best_estimator_

print("Best Logistic Regression Params:", grid_lr.best_params_)
log_best_preds = best_log_model.predict(X_valid_scaled)
print("Tuned Logistic Regression F1:", f1_score(y_valid, log_best_preds))
print("Tuned Logistic Regression AUC-ROC:", roc_auc_score(y_valid, best_log_model.predict_proba(X_valid_scaled)[:, 1]))


--- Logistic Regression ---
Base Logistic Regression F1: 0.3214953271028037
Base Logistic Regression AUC-ROC: 0.7874608044099569
Upsampled Logistic Regression F1: 0.5125541125541125
Upsampled Logistic Regression AUC-ROC: 0.7924179958078265
Best Logistic Regression Params: {'C': 0.1}
Tuned Logistic Regression F1: 0.5125541125541125
Tuned Logistic Regression AUC-ROC: 0.7924365043009112


### 4b. Random Forest — Baseline (No Imbalance Handling)

In [5]:
# Baseline: Random Forest Without Class Imbalance Handling
print("\n--- Random Forest (Baseline - No Imbalance Handling) ---")

# Train on original imbalanced data without any imbalance handling
rf_baseline = RandomForestClassifier(random_state=12345)
rf_baseline.fit(X_train_scaled, y_train)

# Validate performance
rf_baseline_preds = rf_baseline.predict(X_valid_scaled)
print("Baseline Random Forest F1:", f1_score(y_valid, rf_baseline_preds))
print("Baseline Random Forest AUC-ROC:", roc_auc_score(y_valid, rf_baseline.predict_proba(X_valid_scaled)[:, 1]))

# Hyperparameter tuning
param_grid_baseline = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3]
}

grid_baseline = GridSearchCV(
    RandomForestClassifier(random_state=12345),
    param_grid_baseline,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

grid_baseline.fit(X_train_scaled, y_train)
best_baseline_rf = grid_baseline.best_estimator_

rf_baseline_tuned_preds = best_baseline_rf.predict(X_valid_scaled)
print("Best Baseline RF Params:", grid_baseline.best_params_)
print("Tuned Baseline RF F1:", f1_score(y_valid, rf_baseline_tuned_preds))
print("Tuned Baseline RF AUC-ROC:", roc_auc_score(y_valid, best_baseline_rf.predict_proba(X_valid_scaled)[:, 1]))


--- Random Forest (Baseline - No Imbalance Handling) ---
Baseline Random Forest F1: 0.5555555555555556
Baseline Random Forest AUC-ROC: 0.8534003957732772
Best Baseline RF Params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Tuned Baseline RF F1: 0.5656877897990726
Tuned Baseline RF AUC-ROC: 0.8553669231635334


### 4c. Random Forest — Class Weighting

In [6]:
# Model 2: Random Forest with Class Weighting

print("\n--- Random Forest (Class Weighting) ---")

# Train on original imbalanced data with balanced class weights
rf_model_weighted = RandomForestClassifier(class_weight='balanced', random_state=12345)
rf_model_weighted.fit(X_train_scaled, y_train)

rf_preds_weighted = rf_model_weighted.predict(X_valid_scaled)
print("Weighted Random Forest F1:", f1_score(y_valid, rf_preds_weighted))
print("Weighted Random Forest AUC-ROC:", roc_auc_score(y_valid, rf_model_weighted.predict_proba(X_valid_scaled)[:, 1]))

# Hyperparameter tuning
param_grid_weighted = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3],
    'class_weight': ['balanced']
}

grid_weighted = GridSearchCV(
    RandomForestClassifier(random_state=12345),
    param_grid_weighted,
    scoring='f1',
    cv=3,
    n_jobs=-1
)
grid_weighted.fit(X_train_scaled, y_train)
best_weighted_rf = grid_weighted.best_estimator_

rf_weighted_preds = best_weighted_rf.predict(X_valid_scaled)
print("Best Weighted RF Params:", grid_weighted.best_params_)
print("Tuned Weighted RF F1:", f1_score(y_valid, rf_weighted_preds))
print("Tuned Weighted RF AUC-ROC:", roc_auc_score(y_valid, best_weighted_rf.predict_proba(X_valid_scaled)[:, 1]))

# Threshold optimization
probs_valid = best_weighted_rf.predict_proba(X_valid_scaled)[:, 1]
best_threshold = 0.5
best_f1 = 0

for t in [x / 100 for x in range(20, 80)]:
    preds = (probs_valid > t).astype(int)
    score = f1_score(y_valid, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"\nOptimal threshold based on validation: {best_threshold:.2f}")
print(f"Best validation F1 with threshold: {best_f1:.3f}")


--- Random Forest (Class Weighting) ---
Weighted Random Forest F1: 0.5522620904836193
Weighted Random Forest AUC-ROC: 0.8493747985273408
Best Weighted RF Params: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 200}
Tuned Weighted RF F1: 0.6293888166449935
Tuned Weighted RF AUC-ROC: 0.8671028501536977

Optimal threshold based on validation: 0.42
Best validation F1 with threshold: 0.647


### 4d. Random Forest — Upsampling

In [7]:
# Model 2: Random Forest with Upsampling

print("\n--- Random Forest (Upsampling) ---")

# Train on upsampled data
rf_model_up = RandomForestClassifier(random_state=12345)
rf_model_up.fit(X_train_up, y_train_up)

rf_preds_up = rf_model_up.predict(X_valid_scaled)
print("Upsampled Random Forest F1:", f1_score(y_valid, rf_preds_up))
print("Upsampled Random Forest AUC-ROC:", roc_auc_score(y_valid, rf_model_up.predict_proba(X_valid_scaled)[:, 1]))

# Hyperparameter tuning
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=12345),
    param_grid_rf,
    scoring='f1',
    cv=3,
    n_jobs=-1
)
grid_rf.fit(X_train_up, y_train_up)
best_rf = grid_rf.best_estimator_

rf_best_preds = best_rf.predict(X_valid_scaled)
print("Best Random Forest Params:", grid_rf.best_params_)
print("Tuned Random Forest F1:", f1_score(y_valid, rf_best_preds))
print("Tuned Random Forest AUC-ROC:", roc_auc_score(y_valid, best_rf.predict_proba(X_valid_scaled)[:, 1]))

# Threshold optimization
probs_valid = best_rf.predict_proba(X_valid_scaled)[:, 1]
best_threshold = 0.5
best_f1 = 0

for t in [x / 100 for x in range(20, 80)]:
    preds = (probs_valid > t).astype(int)
    score = f1_score(y_valid, preds)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

# Calculate F1 scores for comparison
best_weighted_rf_f1 = f1_score(y_valid, best_weighted_rf.predict(X_valid_scaled))
best_upsampled_rf_f1 = f1_score(y_valid, best_rf.predict(X_valid_scaled))

print(f"\nOptimal threshold based on validation: {best_threshold:.2f}")
print(f"Best validation F1 with threshold: {best_f1:.3f}")

print("\n=== APPROACH COMPARISON ===")
print(f"Class Weighting F1: {best_weighted_rf_f1:.3f}")
print(f"Upsampling F1: {best_upsampled_rf_f1:.3f}")


--- Random Forest (Upsampling) ---
Upsampled Random Forest F1: 0.5938375350140056
Upsampled Random Forest AUC-ROC: 0.8564188225205175
Best Random Forest Params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 300}
Tuned Random Forest F1: 0.5949720670391062
Tuned Random Forest AUC-ROC: 0.8580329173549512

Optimal threshold based on validation: 0.40
Best validation F1 with threshold: 0.626

=== APPROACH COMPARISON ===
Class Weighting F1: 0.629
Upsampling F1: 0.595


## 5. Model Performance Comparison

| Model | Approach | Validation F1 | Validation AUC-ROC |
|---|---|---|---|
| Logistic Regression | Upsampled + Tuned | 0.513 | 0.792 |
| Random Forest | Baseline (no handling) | 0.566 | 0.855 |
| Random Forest | Class Weighting | **0.629** | **0.867** |
| Random Forest | Upsampling | 0.595 | 0.858 |

### Key Insights

- **Class weighting** performed best for F1, achieving **0.629** — significantly above the required 0.59 threshold.
- The **baseline Random Forest** showed surprisingly strong performance (F1 = 0.566), demonstrating the model's inherent robustness to class imbalance.
- **Upsampling** was moderately effective (F1 = 0.595), outperforming the baseline but falling short of class weighting.
- All approaches achieved strong AUC-ROC scores (0.84–0.87), indicating reliable ranking ability regardless of imbalance strategy.
- Both imbalance correction methods improved F1 over the baseline, with class weighting providing the largest gain (+0.063 F1 points).

## 6. Best Model Selection

In [8]:
# Model Comparison

log_f1 = f1_score(y_valid, log_best_preds)
rf_f1 = f1_score(y_valid, rf_best_preds)

if rf_f1 > log_f1:
    best_model = best_rf
    best_model_name = "Random Forest"
else:
    best_model = best_log_model
    best_model_name = "Logistic Regression"

print(f"\nBest model selected: {best_model_name}")


Best model selected: Random Forest


## 7. Final Test Evaluation

The best model (Random Forest with upsampling and optimized threshold) is evaluated on the held-out test set — data the model has never seen during training or validation.

In [9]:
# Final Test Evaluation

# Evaluate best model with tuned parameters and optimized threshold
probs_test = best_rf.predict_proba(X_test_scaled)[:, 1]
test_preds = (probs_test > best_threshold).astype(int)

# Compute metrics
test_f1 = f1_score(y_test, test_preds)
test_auc = roc_auc_score(y_test, probs_test)

print("\n--- Final Test Evaluation ---")
print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Final Test F1 Score: {test_f1:.3f}")
print(f"Final Test AUC-ROC: {test_auc:.3f}")


--- Final Test Evaluation ---
Optimal Threshold: 0.40
Final Test F1 Score: 0.606
Final Test AUC-ROC: 0.859


## 8. Summary

After preprocessing and addressing class imbalance, two models were trained and evaluated across multiple strategies. The best-performing model was a **Random Forest classifier** trained on upsampled data with threshold optimization.

Final test results:
- **F1 Score: 0.606** — exceeds the required threshold of 0.59
- **AUC-ROC: 0.859** — strong ranking performance across all classification thresholds

The project demonstrates that Random Forest with class imbalance correction (either upsampling or class weighting) is a reliable approach for churn prediction on this dataset.